# Attention U-Net 2D: Test (test-only, load a pre-trained checkpoint)
This is a standalone test notebook, separated from `Source/File_Train/Attention_Unet2D.ipynb` (the original combined train+test notebook). Architecture: a standard Attention U-Net (no prompt, fully automatic), used as the reference for the unconditioned-attention baseline in the ablation table (see the ablation variant with PSG and the original attention gate).

In [ ]:
# =========================================================
# CELL 1 - SETUP (test-only: clone repo, download dataset + trained checkpoint)
# =========================================================
%cd /content
import torch

print("=" * 50)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("=" * 50)

# ===== CLONE REPO (PGA_Unet2D / main) =====
!git clone https://github.com/ThongLuc2k3/PGA_Unet2D.git

# ===== DATASET =====
!gdown --id 1sfMPFQvADmZLCPJC3xPyYDrblnFZ4kQv
!unzip -q dataset_FracAtlas.zip
!mv dataset_FracAtlas PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation/

%cd PGA_Unet2D/Source/Prompt-Guided-XRay-Segmentation
!pip install -q tqdm opencv-python matplotlib scikit-image gdown

# ===== CHECKPOINT (fill in before running) =====
# Upload the checkpoint (`att_unet_best.pth`, see `Source/File_Train/Attention_Unet2D.ipynb`) to
# Google Drive, then fill in the ID here
import os, gdown
CKPT_ID   = '1-70seXLCl-Ez-6j24GUto0IPssgyD2eu'  # att_unet_best.pth
CKPT_PATH = 'checkpoints/att_unet_best.pth'
os.makedirs('checkpoints', exist_ok=True)
assert CKPT_ID, '❌ CKPT_ID is missing - upload the checkpoint to Drive and fill in the ID here'
gdown.download(f'https://drive.google.com/uc?id={CKPT_ID}', CKPT_PATH, quiet=False)
assert os.path.exists(CKPT_PATH)
print(f"\n✅ Checkpoint: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)//1024} KB)")
print("\nSETUP DONE!")


In [ ]:
# @title
# =========================================================
# TEST PHASE (ATTENTION U-NET 2D | ImgSize=512 | Batch=4 | Epochs≤100)
# =========================================================

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import binary_erosion, distance_transform_edt

# Import model and dataset
from models.networks.attention_unet_2D import Attention_UNet_2D

# =========================================================
# CONFIGURATION
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "checkpoints/att_unet_best.pth"
IMG_SIZE = 512  # Note: this notebook uses 512

# =========================================================
# HELPER FUNCTIONS (DENOISING & METRICS) - SHARED PROTOCOL
# =========================================================

def extract_lcc(binary_map: np.ndarray) -> np.ndarray:
    """Remove boundary noise and keep only the largest connected component for fair model comparison."""
    if binary_map.sum() == 0:
        return binary_map
    mask_uint8 = binary_map.astype(np.uint8)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_uint8, connectivity=8)
    if num_labels <= 1:
        return binary_map
    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest_label).astype(np.float32)

def calc_hd95(pred: np.ndarray, gt: np.ndarray) -> float:
    """Compute the 95th-percentile Hausdorff distance"""
    pred, gt = pred.astype(bool), gt.astype(bool)
    if not pred.any() and not gt.any(): return 0.0
    if not pred.any() or not gt.any(): return float(IMG_SIZE)
    pe = pred ^ binary_erosion(pred)
    ge = gt   ^ binary_erosion(gt)
    d1 = distance_transform_edt(~ge)[pe]
    d2 = distance_transform_edt(~pe)[ge]
    if not len(d1) or not len(d2): return float(IMG_SIZE)
    return float(max(np.percentile(d1, 95), np.percentile(d2, 95)))

def calc_cbl(pred_bin: np.ndarray, gt_bin: np.ndarray):
    """Compute Center-Based Localization offset"""
    if gt_bin.sum() == 0: return None
    ys, xs = np.where(gt_bin)
    gt_diag = np.sqrt((ys.max()-ys.min())**2 + (xs.max()-xs.min())**2) + 1e-6
    if pred_bin.sum() == 0: return 0.0
    yp, xp = np.where(pred_bin)
    d = np.sqrt((xp.mean()-xs.mean())**2 + (yp.mean()-ys.mean())**2)
    return float(np.clip(1.0 - d/gt_diag, 0.0, 1.0))

def get_centroid(binary_map: np.ndarray):
    """Get centroid coordinates for visualization"""
    if binary_map.sum() == 0: return None, None
    ys, xs = np.where(binary_map)
    return float(xs.mean()), float(ys.mean())

# =========================================================
# LOAD MODEL & DATASET
# =========================================================

class ImageMaskDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, mask_dir, img_size=512, is_train=False):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.img_size = img_size
        self.is_train = is_train
        self.images = sorted([f for f in os.listdir(image_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        self.masks = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith(".png")])

    def __len__(self):
        return min(len(self.images), len(self.masks))

    def _resize_and_pad(self, array, interpolation, pad_value=0):
        h, w = array.shape[:2]
        scale = min(self.img_size / w, self.img_size / h)
        new_w = max(1, int(round(w * scale)))
        new_h = max(1, int(round(h * scale)))
        resized = cv2.resize(array, (new_w, new_h), interpolation=interpolation)
        padded = np.full((self.img_size, self.img_size), pad_value, dtype=resized.dtype)
        pad_left = (self.img_size - new_w) // 2
        pad_top = (self.img_size - new_h) // 2
        padded[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
        return padded

    def __getitem__(self, idx):
        image = cv2.imread(os.path.join(self.image_dir, self.images[idx]), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(os.path.join(self.mask_dir, self.masks[idx]), cv2.IMREAD_GRAYSCALE)
        image = self._resize_and_pad(image, cv2.INTER_LINEAR, pad_value=0)
        mask = self._resize_and_pad(mask, cv2.INTER_NEAREST, pad_value=0)
        image = (image.astype(np.float32) / 255.0 - 0.5) / 0.5
        mask = (mask > 127).astype(np.float32)
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)
        return image, mask

model = Attention_UNet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True))
model.eval()

test_dataset = ImageMaskDataset(
    image_dir="dataset_FracAtlas/test/images",
    mask_dir="dataset_FracAtlas/test/masks",
    img_size=IMG_SIZE,
    is_train=False
)
test_dataset.images = sorted(test_dataset.images)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# =========================================================
# TEST LOOP
# =========================================================

SHOW_INDEX = list(range(10))  # First 10 images in order
all_dice, all_iou, all_pre, all_rec, all_hd95, all_cbl = [], [], [], [], [], []
smooth = 1e-5

with torch.no_grad():
    for idx, (images, masks) in enumerate(test_loader):
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        outputs = model(images)
        preds = (torch.sigmoid(outputs) > 0.5).float()

        # Convert to NumPy
        img_np = (images[0, 0].cpu().numpy() + 1) / 2.0  # Restore intensity after normalization for display
        gm = masks[0, 0].cpu().numpy()
        pm = preds[0, 0].cpu().numpy()

        # POST-PROCESSING (Remove small noisy fragments for stable HD95 computation)
        pm = extract_lcc(pm)

        # Compute the four primary metrics
        tp = (pm * gm).sum()
        fp = (pm * (1 - gm)).sum()
        fn = ((1 - pm) * gm).sum()

        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        iou  = (tp + smooth) / (tp + fp + fn + smooth)
        pre  = (tp + smooth) / (tp + fp + smooth)
        rec  = (tp + smooth) / (tp + fn + smooth)

        # Compute distance and localization metrics
        hd = calc_hd95(pm.astype(bool), gm.astype(bool))
        cbl = calc_cbl(pm.astype(bool), gm.astype(bool))

        all_dice.append(dice)
        all_iou.append(iou)
        all_pre.append(pre)
        all_rec.append(rec)
        all_hd95.append(hd)
        if cbl is not None:
            all_cbl.append(cbl)

        # =====================================================
        # VISUALIZATION (4 COLUMNS - MATCHED TO EXPERIMENT A/B)
        # =====================================================
        if idx in SHOW_INDEX:
            cx_gt, cy_gt = get_centroid(gm)
            cx_p, cy_p   = get_centroid(pm)

            fig, axes = plt.subplots(1, 4, figsize=(20, 5))
            fig.suptitle(f"Attention U-Net test | Sample ID: {idx}", fontsize=14, fontweight='bold')

            # 1. Input image
            axes[0].imshow(img_np, cmap='gray')
            axes[0].set_title("Input image", fontsize=11, fontweight='bold')

            # 2. Ground truth + centroid
            axes[1].imshow(img_np, cmap='gray')
            green = np.zeros((*gm.shape, 4), dtype=np.float32)
            green[gm == 1] = [0, 1, 0, 0.35]
            axes[1].imshow(green)
            if gm.max() > 0:
                axes[1].contour(gm, [0.5], colors='lime', linewidths=1.5)
            if cx_gt is not None:
                axes[1].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, markeredgecolor='black', label='GT centroid')
                axes[1].legend(loc='lower right', fontsize=8)
            axes[1].set_title("Ground truth", fontsize=11, fontweight='bold')

            # 3. Prediction + centroid
            axes[2].imshow(img_np, cmap='gray')
            red = np.zeros((*pm.shape, 4), dtype=np.float32)
            red[pm == 1] = [1, 0, 0, 0.35]
            axes[2].imshow(red)
            if pm.max() > 0:
                axes[2].contour(pm, [0.5], colors='red', linewidths=1.5)
            if cx_p is not None:
                axes[2].plot(cx_p, cy_p, 'o', color='red', ms=8, markeredgecolor='white', label='Predicted centroid')
                axes[2].legend(loc='lower right', fontsize=8)
            axes[2].set_title("Prediction (Att-UNet)", fontsize=11, fontweight='bold')

            # 4. Overlay comparison + metrics (with centroid link)
            axes[3].imshow(img_np, cmap='gray')
            if gm.max() > 0: axes[3].contour(gm, [0.5], colors='lime', linewidths=2)
            if pm.max() > 0: axes[3].contour(pm, [0.5], colors='red', linewidths=2, linestyles='--')
            if cx_gt is not None: axes[3].plot(cx_gt, cy_gt, 'o', color='lime', ms=8, markeredgecolor='black')
            if cx_p is not None:  axes[3].plot(cx_p, cy_p, 'o', color='red', ms=8, markeredgecolor='white')

            # Draw the centroid link to visualize localization offset
            if cx_gt is not None and cx_p is not None:
                axes[3].plot([cx_gt, cx_p], [cy_gt, cy_p], '--', color='yellow', lw=1.5)

            title_str = (f"Dice: {dice:.3f} | IoU: {iou:.3f} | HD95: {hd:.1f}px\n"
                         f"CBL: {cbl:.3f} | Pre: {pre:.3f} | Rec: {rec:.3f}")
            axes[3].set_title(title_str, fontsize=10, fontweight='bold')

            for ax in axes: ax.axis('off')
            plt.tight_layout()
            plt.show()

# =========================================================
# FINAL REPORT (THESIS SUMMARY)
# =========================================================
print("\n" + "=" * 60)
print("📊 FINAL TEST RESULTS - ATTENTION U-NET")
print("=" * 60)
print(f"Mean Dice ↑      : {np.mean(all_dice):.4f}")
print(f"Mean IoU ↑       : {np.mean(all_iou):.4f}")
print(f"Mean Precision ↑ : {np.mean(all_pre):.4f}")
print(f"Mean Recall ↑    : {np.mean(all_rec):.4f}")
print(f"Mean HD95 ↓ (px) : {np.mean(all_hd95):.2f}")
print(f"Mean CBL ↑       : {np.mean(all_cbl):.4f}")
print(f"Total Samples    : {len(all_dice)}")
print("=" * 60)
# =========================================================
# SAVE SUMMARY CSV
# =========================================================
import csv, os
os.makedirs("results", exist_ok=True)
csv_path = "results/attunet2d_results.csv"
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["model", "dice", "iou", "precision", "recall", "hd95", "cbl", "n_samples"])
    writer.writerow([
        "AttUNet2D",
        f"{np.mean(all_dice):.4f}",
        f"{np.mean(all_iou):.4f}",
        f"{np.mean(all_pre):.4f}",
        f"{np.mean(all_rec):.4f}",
        f"{np.mean(all_hd95):.4f}",
        f"{np.mean(all_cbl):.4f}",
        len(all_dice)
    ])
print(f"\nResults saved: {csv_path}")

# =========================================================
# SAVE PER-IMAGE CSV (for optional statistical testing)
# =========================================================
img_names = [os.path.basename(p) for p in test_dataset.images]
per_image_path = "results/attunet2d_per_image.csv"
with open(per_image_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["img_name", "dice", "iou", "precision", "recall", "hd95"])
    for i in range(len(all_dice)):
        writer.writerow([
            img_names[i] if i < len(img_names) else f"idx_{i}",
            f"{all_dice[i]:.6f}", f"{all_iou[i]:.6f}",
            f"{all_pre[i]:.6f}", f"{all_rec[i]:.6f}", f"{all_hd95[i]:.6f}"
        ])
print(f"Per-image results saved: {per_image_path}")
